# **Extreme Water Levels: Station Exploration**

Last updated on: 08/14/2026

## Overview
NOAA's Center for Operational Oceanographic Products and Services (CO-OPS) is the nation's source for [coastal inundation data](https://tidesandcurrents.noaa.gov) through its network of long-term water level gauges. In this notebook, we will review, plot, and examine CO-OPS's [Generalized Extreme Value (GEV) analysis](https://tidesandcurrents.noaa.gov/est) and regional frequency analysis (RFA) of tide gauge data fit with a Generalized Pareto Distribution (GPD) from the [2022 Sea Level Rise Technical Report](https://oceanservice.noaa.gov/hazards/sealevelrise/sealevelrise-tech-report.html?utm_medium=email&utm_source=GovDelivery). This notebook will walk through different ways extreme water levels are calculated and how different graphics that are part of CO-OPS's Extremes product are produced.

This notebook is designed to explore a single station. Users can choose their station of interest here: [Extreme Water Levels Map](https://tidesandcurrents.noaa.gov/est/).

For this example we will use [Sewells Point, VA (8638610)](https://tidesandcurrents.noaa.gov/est/est_station.shtml?stnid=8638610).

## Prerequisites and running the Notebook
This notebook uses Python programming and users may run it either in **Google Colab** or **Jupyter Notebook**.

* **Jupyter Notebook:** Recommended if you prefer to work offline and manage libraries and dependencies personally.
* **Google Colab:** The best option for cloud-based instant access, seamless cooperation, and if you have a good internet connection.

This notebook is designed so that people with limited familiarity with Python can run it.

### Using Google Colab

There are two ways to open this notebook in Colab.

**Method 1: Open directly from GitHub (Recommended)**

This is the easiest way to *open* the notebook, but **you will still need to download the required `.csv` files** separately.

1.  Open [Google Colab](https://colab.research.google.com/).
2.  In the menu bar, go to **File > Open Notebook**.
3.  In the popup window, click on the **GitHub** tab.
4.  Paste the repository URL into the search bar and press Enter:
    `https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks`
5.  Click on `Extremes_Station_Exploration_Notebook/Extremes_Station_Exploration_Notebook.ipynb` to open it.

**Method 2: Download and Upload (All Files)**

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer. This folder contains the notebook (`.ipynb`) and the data files (`.csv`).
4.  Open [Google Colab](https://colab.research.google.com/).
5.  Go to **File > Upload notebook...** and select the `Extremes_Station_Exploration_Notebook.ipynb` file from the folder you just extracted.

### Using Jupyter Notebook

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer.
4.  Start Jupyter Notebook on your local machine and navigate to the extracted folder (`.../Extremes_Station_Exploration_Notebook/`) to open `Extremes_Station_Exploration_Notebook.ipynb`.

### ▶️ Running the Notebook

Once you have the notebook open:

1.  [Explore different station/s](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html) and decide which you want to analyze.
2.  In **Step 3** of the notebook, change the **station variable** and **unit of interest**.
3.  In the top menu, click **Runtime** and choose **Run All**.
4.  When the 'Choose Files' box in **Step 2** becomes active, click on it.
5.  Upload the `ex_params_2025.csv` file that you downloaded.
6.  You will see a green check mark to the left of each cell as it is successfully executed.

Users should be prepared to commit 1-2 hours to familiarize themselves with the notebook, upload data, run code and examine results.

## Learning Outcomes
In this notebook users will be able to:
  1.  Explore past extreme events identified in GEV/RFA analysis,
  2.  Learn more about probability curves and identifying the probability of different water levels occuring at a station
  3.  Examine future risk curves and how extreme water levels can change in the future.

## Data Source and Documentations
Input and output documentation can be found in the [CO-OPS Data API](https://api.tidesandcurrents.noaa.gov/api/prod/), [CO-OPS Derived Product API v0.1](https://api.tidesandcurrents.noaa.gov/dpapi/prod/) and the [CO-OPS Metadata API](https://api.tidesandcurrents.noaa.gov/mdapi/prod/).

Standard templates are available within the [CO-OPS API URL Builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html), but this notebook is designed to give you more flexiblity through building our own API template to generate queries from our Data API, MetaData API, and Derived Product API.

All data used for this notebook come from the CO-OPS API, except for the location, scale, and shape parameters from the CO-OPS GEV analysis which is provided in a csv file.


## Software
This tutorial uses the following Python packages:
- datetime
- requests
- time
- pandas
- numpy
- matplotlib
- math
- plotly
- warnings
- noaa-coops

## **Step 1**
## Install and Load Packages
Lets check if you have these packages installed. If not, this cell will install them.

In [57]:
# @title
import subprocess, sys
packages = ["datetime","requests","time","pandas","numpy","matplotlib","warnings","math","plotly","re","json", "noaa-coops"]
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

Now we load our packages and assign them names to use throughout the notebook.

In [58]:
# @title
import datetime as dt
from datetime import timedelta
import requests
import time as time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import math
from math import exp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import json
from noaa_coops import Station

#this is to suppress user warnings
warnings.filterwarnings("ignore")

## **Step 2**
## Upload External Data
Here we will upload the full set of CO-OPS exceedance parameters (ex_params_2025.csv). These files can be found on the CO-OPS [Github](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks/tree/main/Extremes_Station_Exploration_Notebook) page in the Coastal_Hazards_Example_Notebooks repository with this notebook.

In [59]:
try:
    from google.colab import files
    uploaded = files.upload()
except ModuleNotFoundError:
    print("User running script locally")


## **Step 3**
## Here we choose the station of interest and measurement unit i.e. 'english' (feet) or 'metric' (meter)
Click [here](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html) to see a map of stations to get Station IDs

- Note that stations with limited data in the [National Tidal Datum Epoch (NTDE)](https://tidesandcurrents.noaa.gov/datum_options.html) 1983-2001 will not work in this notebook

### The stations with the longest periods of record (greater than 95 years) are:
|Station ID and Name  | Station ID and Name | Station ID and Name|
|:--------------------|:--------------------|:--------------------|
|1612340	Honolulu, HI	| 8574680	Baltimore, MD	| 8771450	Galveston, TX |
|1617760	Hilo, HI	| 8575512	Annapolis, MD	| 9410170	San Diego, CA |
|8410140	Eastport, ME	|	8594900	Washington, DC	| 9410230	La Jolla, CA|
|8418150	Portland, ME	|	8638610	Sewells Point, VA	| 9410660	Los Angeles, CA|
|8443970	Boston, MA	| 8665530	Charleston, SC	| 9414290 San Francisco, CA|
|8452660	Newport, RI	| 8720030	Fernandina Beach, FL | 9439040	Astoria, OR|
|8518750	The Battery, NY	|	8724580	Key West, FL| 9447130	Seattle, WA|
|8534720	Atlantic City, NJ	|	8727520	Cedar Key, FL | 9450460	Ketchikan, AK|
|8557380	Lewes, DE	| 8729840	Pensacola, FL| 9451600	Sitka, AK |

In [187]:
#input station ID below
station = '8638610'

#options are metric or english
units = 'metric'

## **Step 4**
## Next, we pull in our data from the CO-OPS API

In [188]:
# @title Station Metadata
print(f"Fetching data for Station ID: {station}...\n")

# Initialize the station object (this downloads the metadata)
station_meta = Station(station)

# General Metadata
stationName = station_meta.name
print(f"Station Name: {stationName}")
print(f"Coordinates (Lat, Lon): {station_meta.lat_lon}")

station_info = station+' '+stationName

metadata_dict = station_meta.metadata
established_date = metadata_dict.get('details', {}).get('established', 'Unknown')
print(f"Established Date: {established_date}\n")


# Datum & Epoch Information
# Access the specific 'datums' dictionary within the metadata
datums_meta = metadata_dict.get('datums', {})

# Grab the epoch info, defaulting to '1983-2001' if missing or '0000-0000'
epoch_info = datums_meta.get('epoch', '0000-0000')
if epoch_info == '0000-0000' or not epoch_info:
    epoch_info = '1983-2001'

epic_start = int(epoch_info[0:4])
epic_end = int(epoch_info[5:])
epic_length = epic_end - epic_start
epic_mid = int(epic_start + epic_length / 2)

print(f"Epoch: {epic_start}-{epic_end}; Epoch Mid: {epic_mid}")


# Control vs. Subordinate Station
# .strip() removes any accidental spaces from the API response
ctrl_station = datums_meta.get('ctrlStation', '').strip()

if not ctrl_station:
    # If the control station string is empty, this is a Control Station
    print(f"Hierarchy: Station {station} is a Control Station.")
else:
    # If a control station ID exists, this is a Subordinate Station
    print(f"Hierarchy: Station {station} is a Subordinate Station.")
    print(f"Control Station ID: {ctrl_station}")

    # Extract the comparison dates from the DatumAnalysisPeriod list
    analysis_period_list = datums_meta.get('DatumAnalysisPeriod', ['Dates not available'])
    if analysis_period_list:
        analysis_period = analysis_period_list
        print(f"Dates of Comparison: {analysis_period}")



Fetching data for Station ID: 8638610...

Station Name: Sewells Point
Coordinates (Lat, Lon): {'lat': 36.94278, 'lon': -76.32861}
Established Date: 1927-07-01 00:00:00

Epoch: 1983-2001; Epoch Mid: 1992
Hierarchy: Station 8638610 is a Control Station.


In [189]:
# @title RFA results
#getting the RFA analysis results
product = 'extrfa'
datum = 'mhhw'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/'

myurl = server + product + '.json?station=' + station + '&units=' + units + '&datum=' + datum

urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables and make it a dataframe.
myrfa = pd.DataFrame(content['ExtRfa'][0]['ewlProbabilities'])
high_x = myrfa['ci_high'].iloc[-1]
local_u_trend = pd.DataFrame(content['ExtRfa'][0]['localUTrend'])
local_u_station = pd.DataFrame(content['ExtRfa'][0]['localUIndex'])

local_u_trend = local_u_trend['Utrend']/1000
local_u_station = local_u_station['Uindex']

In [190]:
# @title Getting daily max fron Data API which is derived from hourly observations
# Get the current year
current_year = dt.datetime.now().year
start_year_total = 1920
end_year_total = current_year-1

def fetch_daily_data():
    """
    Fetches daily max water level data from the NOAA Tides and Currents API
    for a specified station from 1850 to end of last year, handling the API's 10-year limit.

    Returns:
        pandas.DataFrame: A DataFrame containing all the data, or an empty
                          DataFrame if no data could be fetched.
    """
    # --- Configuration ---
    base_url = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"

    # List to hold each 10-year DataFrame
    all_dfs = []

    print(f"Starting data fetch for station {station} from {start_year_total} to {end_year_total}...")

    # --- Loop through the date range in 10-year chunks ---
    for start_year in range(start_year_total, end_year_total + 1, 10):
        # Determine the end year for this chunk, ensuring it doesn't go past the final end year
        end_year = min(start_year + 9, end_year_total)

        # Format dates for the API URL (YYYYMMDD)
        begin_date = f"{start_year}0101"
        end_date = f"{end_year}1231"

        # --- Construct the API request URL ---
        params = {
            'begin_date': begin_date,
            'end_date': end_date,
            'station': station,
            'product': 'daily_max_min',
            'max_min_type': 'max',
            'datum': 'MHHW',
            'time_zone': 'gmt',
            'units': units,
            'format': 'json',
            'interval': 'h'
        }

        try:
            # --- Make the API request ---
            response = requests.get(base_url, params=params)
            response.raise_for_status()  # This will raise an HTTPError for bad responses (4xx or 5xx)
            print(f"Requesting URL: {response.url}")

            # --- JSON FIX IMPLEMENTATION ---
            try:
                # 1. Attempt to clean the raw text using regex (removes trailing commas before } or ])
                cleaned_text = re.sub(r',\s*([}\]])', r'\1', response.text)
                # 2. Load the cleaned text as JSON
                data = json.loads(cleaned_text)

            except json.JSONDecodeError as e:
                # Catch the specific JSON error if cleaning didn't work or the error is elsewhere
                print(f"JSONDecodeError: {e}")
                print(f"Raw response text (uncleaned): {response.text}")
                # Skip this chunk if decoding fails
                continue
            # --- END JSON FIX IMPLEMENTATION ---

            # Check if the API returned data or an error message
            if 'data' in data:
                # Convert the 'data' part of the JSON into a DataFrame
                temp_df = pd.DataFrame(data['data'])
                all_dfs.append(temp_df)
                print(f"Successfully fetched data for {start_year}-{end_year}. Found {len(temp_df)} records.")
            elif 'error' in data:
                 print(f"API Error for {start_year}-{end_year}: {data['error']['message']}")
            else:
                print(f"No data found for the period {start_year}-{end_year}.")

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for {start_year}-{end_year}: {e}")

        # Wait a moment before the next request
        time.sleep(1)

    # --- Combine all DataFrames into one ---
    if all_dfs:
        print("\nCombining all fetched data into a single DataFrame...")
        daily_max_df = pd.concat(all_dfs, ignore_index=True)
        print("Data combination complete!")
        return daily_max_df
    else:
        print("\nNo data was fetched. Returning an empty DataFrame.")
        return pd.DataFrame()

# --- Run the function and display results ---
if __name__ == "__main__":
    # Ensure station and units variables are defined before running this block
    daily_max_df = fetch_daily_data()

# Combine all the small lists from each row into one single, large list of records.
# The .sum() method on a Series of lists concatenates them all together.
# Note: This assumes daily_max_df is not empty and has the expected column.
if not daily_max_df.empty and 'dailyMaxHourly' in daily_max_df.columns:
    all_records = daily_max_df['dailyMaxHourly'].sum()
    # Use json_normalize to convert the list of dictionaries into a clean DataFrame.
    daily_max_df = pd.json_normalize(all_records)
else:
    print("Could not process daily_max_df: DataFrame is empty or missing 'dailyMaxHourly' column.")

Starting data fetch for station 8638610 from 1920 to 2025...
Requesting URL: https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?begin_date=19200101&end_date=19291231&station=8638610&product=daily_max_min&max_min_type=max&datum=MHHW&time_zone=gmt&units=metric&format=json&interval=h
Successfully fetched data for 1920-1929. Found 1 records.
Requesting URL: https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?begin_date=19300101&end_date=19391231&station=8638610&product=daily_max_min&max_min_type=max&datum=MHHW&time_zone=gmt&units=metric&format=json&interval=h
Successfully fetched data for 1930-1939. Found 1 records.
Requesting URL: https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?begin_date=19400101&end_date=19491231&station=8638610&product=daily_max_min&max_min_type=max&datum=MHHW&time_zone=gmt&units=metric&format=json&interval=h
Successfully fetched data for 1940-1949. Found 1 records.
Requesting URL: https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?begin_d

In [191]:
# @title Getting CO-OPS Exceedance Probabilities from the DPAPI

#Highs
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&details=exceedancelevels&levelType=high&units=' + units
urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables and make it a dataframe.
mydf_hi = pd.DataFrame(content['GEV_HI']['exceedanceLevels'])
mydf_hi['probability'] = mydf_hi['probability']*100

f1_mhhw = mydf_hi[mydf_hi['probability'] == 1.0]['level'].item()
f10_mhhw = mydf_hi[mydf_hi['probability'] == 10.0]['level'].item()
f50_mhhw = mydf_hi[mydf_hi['probability'] == 50.0]['level'].item()
f99_mhhw = mydf_hi[mydf_hi['probability'] == 99.0]['level'].item()

#Lows
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&details=exceedancelevels&levelType=low&units=' + units

urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables and make it a dataframe.
mydf_lo = pd.DataFrame(content['GEV_LO']['exceedanceLevels'])
mydf_lo['probability'] = mydf_lo['probability']*100

f1_mllw = mydf_lo[mydf_lo['probability'] == 1.0]['level'].item()
f10_mllw = mydf_lo[mydf_lo['probability'] == 10.0]['level'].item()
f50_mllw = mydf_lo[mydf_lo['probability'] == 50.0]['level'].item()
f99_mllw = mydf_lo[mydf_lo['probability'] == 99.0]['level'].item()

In [192]:
# @title Getting CO-OPS Exceedance Levels by Month

product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&details=exceedancelevelsmonth&levelType=high&units=' + units
urlResponse = requests.get(myurl)
content = urlResponse.json()

# A list to hold the DataFrame for each month
monthly_dfs = []
# A list of month abbreviations to build the keys
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

# Loop through each month
for i, month_abbr in enumerate(months):
    month_key = f'GEV_HI_{month_abbr}'

    # Check if the key exists in the JSON response
    if month_key in content:
        # Extract the list of exceedance levels
        exceedance_data = content[month_key]['exceedanceLevels']

        # Convert the list to a DataFrame
        df_month = pd.DataFrame(exceedance_data)

        # Add a column for the month number (1 for JAN, 2 for FEB, etc.)
        df_month['month'] = i + 1

        # Add the new DataFrame to our list
        monthly_dfs.append(df_month)

# Concatenate all the monthly DataFrames into a single one
seasonal_hi = pd.concat(monthly_dfs, ignore_index=True)

# --- Code to parse the monthly LOW water level data ---
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&details=exceedancelevelsmonth&levelType=low&units=' + units
urlResponse = requests.get(myurl)
content = urlResponse.json()
# A list to hold the DataFrame for each month
monthly_dfs_lo = []
# A list of month abbreviations to build the keys
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

# Loop through each month
for i, month_abbr in enumerate(months):
    # This is the key change: look for GEV_LO_ instead of GEV_HI_
    month_key = f'GEV_LO_{month_abbr}'

    # Check if the key exists in the JSON response
    if month_key in content:
        # Extract the list of exceedance levels
        exceedance_data = content[month_key]['exceedanceLevels']

        # Convert the list to a DataFrame
        df_month = pd.DataFrame(exceedance_data)

        # Add a column for the month number (1 for JAN, 2 for FEB, etc.)
        df_month['month'] = i + 1

        # Add the new DataFrame to our list
        monthly_dfs_lo.append(df_month)

# Concatenate all the monthly DataFrames into a single one for low levels
seasonal_lo = pd.concat(monthly_dfs_lo, ignore_index=True)

In [193]:
# @title Getting monthly extremes from dpapi
#Highs
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&extremeType=monthlies&levelType=high&units=' + units

urlResponse = requests.get(myurl)
content=urlResponse.json()

df_monthly = pd.DataFrame(content['extremes'])
df_monthly['highest'] = df_monthly['value']

#Lows
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&extremeType=monthlies&levelType=low&units=' + units

urlResponse = requests.get(myurl)
content=urlResponse.json()

df_monthly_lo = pd.DataFrame(content['extremes'])
df_monthly_lo['lowest'] = df_monthly_lo['value']

df_monthly = df_monthly.merge(df_monthly_lo[['year', 'month', 'lowest']], on=['year', 'month'], how='left')

df_monthly['decimal_date'] = df_monthly['year'] + (df_monthly['month'] - 1) / 12

df_monthly = df_monthly[df_monthly['decimal_date'].astype(int) <= end_year_total].copy()


In [194]:
# @title Getting annual extremes from dpapi
#Highs
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&extremeType=annual&levelType=high&units=' + units

urlResponse = requests.get(myurl)
content=urlResponse.json()

df_annual = pd.DataFrame(content['extremes'])
df_annual['highest'] = df_annual['value']

#Lows
product = 'extremewaterlevels'
server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name='

myurl = server + product + '&station=' + station + '&extremeType=annual&levelType=low&units=' + units

urlResponse = requests.get(myurl)
content=urlResponse.json()

df_annual_lo = pd.DataFrame(content['extremes'])
df_annual_lo['lowest'] = df_annual_lo['value']


In [195]:
# @title Grab the POR trend rate used for detrending and plotting
if units == 'metric':
  mytrendrate = content['trend']/1000
  trend_text = str(round(mytrendrate*1000,2)) + ' mm/year'
else:
  mytrendrate = content['trend']/120
  trend_text = str(round(mytrendrate*120,2)) + ' inches/decade'

trendstartdate = content['trendStartDate']
trendenddate = content['trendEndDate']


In [196]:
# @title Additional Datums info
#grab datum info
server = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/'

myurl = (server + station + '/datums.json?units='+units)

urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
epoch_info = content['epoch']
datums_info = pd.DataFrame(content['datums'])

#if we can't find the epoch info, default to NTDE
if epoch_info == '0000-0000':
  if station == '1770000':
    epoch_info = '2011-2019'
  else:
    epoch_info = '1983-2001'

epic_start = int(epoch_info[0:4])
epic_end = int(epoch_info[5:])
epic_length = epic_end - epic_start
epic_mid = int((epic_start + epic_length/2))+0.5

MSL_datum = datums_info[datums_info['name']=='MSL']
MSL_datum = MSL_datum['value'].iloc[0]
MHHW_datum = datums_info[datums_info['name']=='MHHW']
MHHW_datum = MHHW_datum['value'].iloc[0]
MHHW_msl_datum = MHHW_datum-MSL_datum
MLLW_datum = datums_info[datums_info['name']=='MLLW']
MLLW_datum = MLLW_datum['value'].iloc[0]
MLLW_msl_datum = MLLW_datum-MSL_datum
MHW_datum = datums_info[datums_info['name']=='MHW']
MHW_datum = MHW_datum['value'].iloc[0]
MHW_msl_datum = MHW_datum-MSL_datum
MLW_datum = datums_info[datums_info['name']=='MLW']
MLW_datum = MLW_datum['value'].iloc[0]
MLW_msl_datum = MLW_datum-MSL_datum
NAVD88_datum = datums_info[datums_info['name']=='NAVD88']
try:
  NAVD88_datum = NAVD88_datum['value'].iloc[0]
except IndexError:
  NAVD88_datum = np.nan
NAVD88_msl_datum = NAVD88_datum-MSL_datum

f1_mhhw_msl = f1_mhhw - (MSL_datum-MHHW_datum)
f10_mhhw_msl = f10_mhhw - (MSL_datum-MHHW_datum)
f50_mhhw_msl = f50_mhhw - (MSL_datum-MHHW_datum)
f99_mhhw_msl = f99_mhhw - (MSL_datum-MHHW_datum)

f1_mllw_msl = f1_mllw - (MSL_datum-MLLW_datum)
f10_mllw_msl = f10_mllw - (MSL_datum-MLLW_datum)
f50_mllw_msl = f50_mllw - (MSL_datum-MLLW_datum)
f99_mllw_msl = f99_mllw - (MSL_datum-MLLW_datum)


In [197]:
# @title Grabbing Top Ten WLs which are derived from 6 min data. If 6-min data is not available than the top WL are based on hourly observations or high water marks
#add top ten wl

product = "toptenwaterlevels"
server = "https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/.json?name="

myurl = (server + product +"&station=" + str(station) + "&units="+units)

# Use "requests" to get the Monthly MSL Data
urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
mydata = content['topTenWaterLevels']

# Make it a Dataframe
mytop10 = pd.DataFrame(mydata)

mytop10['date'] = pd.to_datetime(mytop10['peakDate'])

mytop10['height'] = pd.to_numeric(mytop10['height'],downcast='float')
mytop10.index = mytop10['date']
mytop10['decimal_date']=mytop10.index.year+(mytop10.index.dayofyear -1)/365.2425

mytop10

,stationId,stationName,peakDate,peakTime,height,datum,category,event,source,mhhw,mhw,mtl,msl,mlw,mllw,navd,date,decimal_date
date,,,,,,,,,,,,,,,,,,
1933-08-23,8638610,"Sewells Point, VA",1933-08-23,00:00,1.604,MHHW,Tropical,Chesapeake-Potomac Hurricane,High Water Mark,None,None,None,None,None,None,None,1933-08-23,1933.640670
2003-09-18,8638610,"Sewells Point, VA",2003-09-18,21:00,1.564,MHHW,Tropical,Hurricane Isabel,Observed Peak Water Level,None,None,None,None,None,None,None,2003-09-18,2003.711856
2009-11-12,8638610,"Sewells Point, VA",2009-11-12,23:24,1.520,MHHW,Tropical,Hurricane Ida,Observed Peak Water Level,None,None,None,None,None,None,None,2009-11-12,2009.862441
2011-08-27,8638610,"Sewells Point, VA",2011-08-27,23:54,1.462,MHHW,Tropical,Hurricane Irene,Observed Peak Water Level,None,None,None,None,None,None,None,2011-08-27,2011.651622
1962-03-07,8638610,"Sewells Point, VA",1962-03-07,15:00,1.360,MHHW,Extra Tropical,Ash Wednesday Storm,Observed Peak Water Level,None,None,None,None,None,None,None,1962-03-07,1962.177964
2012-10-29,8638610,"Sewells Point, VA",2012-10-29,13:18,1.236,MHHW,Tropical,Hurricane Sandy,Observed Peak Water Level,None,None,None,None,None,None,None,2012-10-29,2012.826848
2006-11-22,8638610,"Sewells Point, VA",2006-11-22,15:18,1.223,MHHW,Extra Tropical,November Nor'easter,Observed Peak Water Level,None,None,None,None,None,None,None,2006-11-22,2006.889820
1936-09-18,8638610,"Sewells Point, VA",1936-09-18,15:00,1.207,MHHW,Tropical,Hurricane Thirteen,Observed Peak Water Level,None,None,None,None,None,None,None,1936-09-18,1936.714594
1998-02-05,8638610,"Sewells Point, VA",1998-02-05,09:18,1.171,MHHW,Extra Tropical,Nor easter,Observed Peak Water Level,None,None,None,None,None,None,None,1998-02-05,1998.095827


## Return Period (RP) vs Average Recurrence Interval (ARI) vs Annual Probability of Exceedance (AEP)

What are they? How are they different? Application and use cases of each

All three of these terms are often used in the context of analyzing and predicting natural phenomena such as coastal flood events in our case.

1. **Return Period (RP)**:  The return period represents the average number of years between events of a certain magnitude or intensity. For instance, if a peak flood level has a return period of 100 years, it means that such a flood is expected to occur once every 100 years on average. However, it's essential to understand that this doesn't imply that the event will definitely occur within any 100-year period; rather, it's a statistical average based on an extreme value analysis of historical data. There may be two or more floods reaching that level or none.** Return periods are commonly used in engineering and risk assessment to design infrastructure that can withstand rare but severe events.

2. **Average Recurrence Interval (ARI)**: This is closely related to the return period. It represents the average time period between events (e.g., floods) of a certain magnitude or intensity. For example, if a flood has a 2 year RP, it means there's a 50% chance of that flood level being exceeded in any given year. The ARI turns out to be 1.44 years.  ARI periods can be fractions of a year for lower levels, unlike return periods. A 1.01 year RP turns out to have a ARI of 0.22 years.

3. **Annual Probability of Exceedance (AEP)**: This is the inverse of the RP. It represents the probability that a specific event, such as a peak flood level of a certain magnitude, will be exceeded in any given year. It can never be 100% or greater, that is, an event can't have a 101% chance of happening within a given period.

In summary, return period and average recurrence interval are essentially the same for periods greater than about 2 years, when indicating the average number of years between events. Annual probability of exceedance, on the other hand, represents the likelihood of a specific event occurring in any given 1-year period. Both are crucial for understanding and managing risks associated with natural hazards. We will be plotting exceedance probability levels relative to AEP and ARI in this notebook, but any of the ARI plots could alternatively be transformed to have RP as the x-axis.

Equations to relate the three concepts:

---

$$
AEP = \frac{1}{RP} = 1 - e^{-\frac{1}{ARI}}
$$

---

$$
RP = \frac{1}{AEP} = \frac{1}{(1 - e^{-\frac{1}{ARI}})}
$$

---

$$
ARI = \frac{-1}{\ln(1 - AEP)}   = \frac{-1}{\ln(1 - \frac{1} {RP})}
$$

---

In [198]:
# @title Plotting the Extreme Water Levels plots as seen on CO-OPS [website](https://tidesandcurrents.noaa.gov/trends-and-extremes/main-product.html?tab=extremes&station-id=8638610&plot=relative-sea-level-trends,extreme-water-levels&mapTab=station)
#function to find y-int for plotting linear trend lines
def getYInt(x1, y1,slope):
    y_intercept = slope * (-x1)+y1
    return (y_intercept)
# Create the 'date' column with the day set to 15
df_monthly['date'] = pd.to_datetime(df_monthly[['year', 'month']].assign(day=15))


# Handles cases with a single, continuous trend
# Calculate a single base y-intercept for a trend line that passes through y=0 at the epoch's midpoint.
y_int_base = getYInt(epic_mid, 0, mytrendrate)

# Calculate the base linear trend across all dates.
df_monthly['linear_trend'] = mytrendrate * df_monthly['decimal_date'] + y_int_base
df_monthly.loc[df_monthly['date'] < trendstartdate, 'linear_trend'] = np.nan
df_monthly.loc[df_monthly['date'] > trendenddate, 'linear_trend'] = np.nan
df_monthly['trend_value'] = df_monthly['linear_trend']
# Add the specific AEP level at the epoch to the base trend.
df_monthly['f1_mhhw_linear'] = df_monthly['linear_trend'] + f1_mhhw
df_monthly['f10_mhhw_linear'] = df_monthly['linear_trend'] + f10_mhhw
df_monthly['f50_mhhw_linear'] = df_monthly['linear_trend'] + f50_mhhw
df_monthly['f99_mhhw_linear'] = df_monthly['linear_trend'] + f99_mhhw


# Get the current year automatically
current_year = dt.date.today().year

# The last row of the DataFrame now holds the projected values for the current month
latest_projections = df_monthly.iloc[-1]

# Assign the final values from the trendline columns
f1_current = latest_projections.get('f1_mhhw_linear')
f10_current = latest_projections.get('f10_mhhw_linear')
f50_current = latest_projections.get('f50_mhhw_linear')
f99_current = latest_projections.get('f99_mhhw_linear')

# Assign the final values from the trendline columns
f1_mllw_current = latest_projections.get('f1_mllw_linear')
f10_mllw_current = latest_projections.get('f10_mllw_linear')
f50_mllw_current = latest_projections.get('f50_mllw_linear')
f99_mllw_current = latest_projections.get('f99_mllw_linear')

# Get the single, most recent trend value from that row
latest_trend_value = latest_projections.get('trend_value')

# Check if latest_trend_value was found before proceeding
if latest_trend_value is not None:
    # Now, calculate the final projected values using this single trend value and your offsets
    f1_current_msl = latest_trend_value + f1_mhhw_msl
    f10_current_msl = latest_trend_value + f10_mhhw_msl
    f50_current_msl = latest_trend_value + f50_mhhw_msl
    f99_current_msl = latest_trend_value + f99_mhhw_msl

    f1_mllw_current_msl = latest_trend_value + f1_mllw_msl
    f10_mllw_current_msl = latest_trend_value + f10_mllw_msl
    f50_mllw_current_msl = latest_trend_value + f50_mllw_msl
    f99_mllw_current_msl = latest_trend_value + f99_mllw_msl

    mhhw_current = latest_trend_value + MHHW_msl_datum
    mllw_current = latest_trend_value + MLLW_msl_datum
    mhw_current = latest_trend_value + MHW_msl_datum
    mlw_current = latest_trend_value + MLW_msl_datum
    msl_current = latest_trend_value + MSL_datum
    navd88_current = latest_trend_value + NAVD88_msl_datum
else:
    print("Warning: Could not find 'trend_value' in the latest projection.")


## **Step 5**
## Exploring CO-OPS Extreme Water Levels

The [Extreme Water Levels Product](https://tidesandcurrents.noaa.gov/est/) provides results of extreme value analyses for 125 NOAA tide gauges locations with at least 30 years of water level data. This product uses monthly extreme levels adjusted for the relative sea level trend which produces an unbiased data set from which the annual highest and lowest water levels are derived. A Generalized Extreme Value (GEV) analysis is then applied to this data set to determine the expected frequency of extremes tides rising above or falling below any given level. The results are a set of annual exceedance probability levels relative to the tidal or geodetic datums for each station.

In generalized extreme value (GEV) analysis, there are three parameters used to describe the distribution of extreme values. The distribution function G(x) of the GEV distribution is given by:


\begin{equation*}
G(x) = \exp\left\{ -\left[1 + \frac{\xi(x - \mu)}{\sigma}\right]^{-\frac{1}{\xi}} \right\}
\end{equation*}

where:

**𝜇** is the location parameter, which determines the location of the distribution along the y-axis. It represents the "center" of the distribution. It shifts the distribution horizontally along the x-axis. For example, if 𝜇 increases, the entire distribution shifts upward, and vice versa.

**𝜎** is the scale parameter, which governs the spread or variability of the distribution. It essentially controls the "range" of the distribution. It shifts the distribution horizontally along the x-axis. When 𝜎 increases, the spread of the distribution widens, and vice versa.

**𝜉** is the shape parameter, which characterizes the shape of the distribution. Positive 𝜉 values indicate a heavy-tailed distribution (fat-tailed), often seen in situations with rare extreme events. A shape parameter of zero (𝜉 = 0) corresponds to an exponential distribution. Negative 𝜉 values indicate a bounded distribution.
The extreme value distributions are sometimes referred to as a Frechet distribution (𝜉 > 0), Gumbel distribution (𝜉 = 0), or Weibull distribution (𝜉 < 0) depending on the sign of 𝜉.


## Annual Exceedance Probability Curves

Annual exceedance probability curves, represented by the bolded black curve, measured in meters or feet relative to the Mean Higher High Water (MHHW), Mean Lower Low Water (MLLW), or Mean Sea Level (MSL) datums, along with 90% confidence intervals. Blue dots represent annual highest or lowest water levels after removal of the Mean Sea Level trend.


In [199]:
# @title

df_annual_sorted = df_annual.sort_values(by='value', ascending=False).copy()
n = len(df_annual_sorted)
# Create rank column
df_annual_sorted['rank'] = np.arange(1, n + 1)
# Calculate exceedance probability in percent
df_annual_sorted['probability'] = (df_annual_sorted['rank'] / (n + 1)) * 100

# --- Directly assign data for plotting from the DataFrame ---
probabilities = mydf_hi['probability'].tolist()
exceedance_levels = mydf_hi['level'].tolist()
upper_ci_levels = mydf_hi['upper90CI'].tolist()
lower_ci_levels = mydf_hi['lower90CI'].tolist()

# Find the maximum value for the y-axis to draw vertical lines
max_level_for_plot = mydf_hi['upper90CI'].max()

# Create figure
fig = go.Figure()

# Create the confidence interval band (the filled area)
fig.add_trace(go.Scatter(
    x=probabilities + probabilities[::-1],  # x-values forward then reverse
    y=upper_ci_levels + lower_ci_levels[::-1],  # y-values forward then reverse
    fill='toself',
    opacity=0.5,
    fillcolor='lightgray',
    line=dict(color='rgba(255,255,255,0)'), # Make the line itself invisible
    showlegend=False,
    name='90% Confidence Interval'
))

# Add the observed annual data as points
fig.add_trace(go.Scatter(
    x=df_annual_sorted['probability'],
    y=df_annual_sorted['value'],
    mode='markers',
    marker=dict(color='royalblue', symbol='circle', size=6),
    name='Observed Annual Maxima'
))

# Plot the main GEV Exceedance Level line and markers
fig.add_trace(go.Scatter(x=probabilities, y=exceedance_levels, mode='lines', line=dict(color='black'), name='GEV Exceedance Level'))
fig.add_trace(go.Scatter(x=probabilities, y=exceedance_levels, mode='lines', line=dict(color='black'), name='GEV Data Points', showlegend=False))

# Add vertical lines for key probabilities
for prob, color, label in [(1.0, 'red', '1% Annual Probability'),
                           (10, 'orange', '10% Annual Probability'),
                           (50, 'blue', '50% Annual Probability'),
                           (99, 'purple','99% Annual Probability')]:
    fig.add_trace(go.Scatter(
        x=[prob, prob],
        y=[0, max_level_for_plot],
        mode='lines',
        line=dict(color=color, width=2),
        showlegend=True,
        name=label
    ))

# Update axis titles based on selected units
yaxis_title = 'feet above MHHW' if units == 'english' else 'meters above MHHW'

fig.update_layout(
    title=f'CO-OPS Annual Exceedance Probability Curve<br>{station_info}',
    xaxis=dict(
        title='Annual Probability of Exceedance (%)',
        type='log',
        autorange='reversed',
        tickangle= 0,
        tickvals=[100,90,80,70,60,50,40,30,20,10,9,8,7,6,5,4,3,2,1],
        ticktext=['100','90','80','70','60','50','40','30','20','10','9','8','7','6','5','4','3','2','1']
    ),
    yaxis=dict(
        title=yaxis_title,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    legend=dict(x=1.01, y=0.95),
    margin=dict(l=50, r=20, t=60, b=50),
    showlegend=True,
    hovermode='x unified', # Improved hover experience
    hoverlabel=dict(bgcolor="white", font_size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=300,
    title_x=0.5
)

fig.show()

## Monthly Extreme Water Levels

Monthly highest and lowest water levels, along with 1%, 10%, 50%, and 99% annual exceedance probability levels shown in red, orange, blue, and purple, respectively. Values are displayed in meters and feet and are relative to Mean Higher High Water (MHHW), Mean Lower Low Water (MLLW), or Mean Sea Level (MSL) datums.

In [200]:
# @title

epic_box_x = [epic_start, epic_end]  # X-axis range for the box
epic_box_y = [0, df_monthly['f1_mhhw_linear'].max()]

fig = go.Figure()

fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['highest'], mode='lines', name='monthly maxima', line=dict(color='cornflowerblue')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f1_mhhw_linear'], mode='lines', name='1% AEP', line=dict(color='red')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f10_mhhw_linear'], mode='lines', name='10% AEP', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f50_mhhw_linear'], mode='lines', name='50% AEP', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f99_mhhw_linear'], mode='lines', name='99% AEP', line=dict(color='purple')))

fig.add_trace(go.Scatter(x=[epic_mid], y=[f1_mhhw], mode='markers', name=epoch_info+' 1% AEP level', marker=dict(symbol='diamond', size=8, color='red', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f10_mhhw], mode='markers', name=epoch_info+' 10% AEP level', marker=dict(symbol='square', size=8, color='orange', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f50_mhhw], mode='markers', name=epoch_info+' 50% AEP level', marker=dict(symbol='triangle-up', size=8, color='blue', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f99_mhhw], mode='markers', name=epoch_info+' 99% AEP level', marker=dict(symbol='circle', size=8, color='purple', line=dict(color='black', width=1)),showlegend=False))

# Add markers for the current year's projected AEP levels
fig.add_trace(go.Scatter(x=[current_year], y=[f1_current], mode='markers', name=f'{current_year} 1% AEP level', marker=dict(symbol='diamond', size=12, color='red', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[current_year], y=[f10_current], mode='markers', name=f'{current_year} 10% AEP level', marker=dict(symbol='square', size=12, color='orange', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[current_year], y=[f50_current], mode='markers', name=f'{current_year} 50% AEP level', marker=dict(symbol='triangle-up', size=12, color='blue', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[current_year], y=[f99_current], mode='markers', name=f'{current_year} 99% AEP level', marker=dict(symbol='circle', size=12, color='purple', line=dict(color='black', width=1)),showlegend=False))

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')
fig.update_layout(
    title={
        'text': 'Monthly Extreme Water Levels<br>'+station_info+'<br>'+trend_text,
        'x': 0.5,
        'y': 0.95,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis=dict(title='Year'),
    yaxis=dict(title='Meters above MHHW'),
    legend=dict(x=1, y=0.5, orientation='v'),
    margin=dict(l=0, r=0, t=50, b=0),
    showlegend=True,
    hovermode='closest',
    hoverlabel=dict(bgcolor="white", font_size=12),
    plot_bgcolor='rgba(255, 255, 255, 1)',  # White gridded background
    paper_bgcolor='rgba(255, 255, 255, 1)',
    width=900,  # Set width of the plot
    height=300,
    title_x=0.5,
    shapes=[
        dict(
            type="rect",
            xref="x",
            yref="y",
            x0=epic_box_x[0],
            y0=epic_box_y[0],
            x1=epic_box_x[1],
            y1=epic_box_y[1],
            fillcolor="gray",
            opacity=0.3,
            layer="below",
            line=dict(color="gray", width=1),
        )]
)

if units == 'english':
  fig.update_layout(yaxis=dict(title='feet above MHHW'))

# Add dummy trace for legend entry of the shape
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=0),
    showlegend=True,
    legendgroup="shape_legend",
    name="Station Epoch",  # Name displayed in legend
    line=dict(color="gray", width=1),
    legendrank=1,
    fill='toself',  # Filled area of the shape
    fillcolor='gray',  # Fill color of the shape
))

fig.show()


In [201]:
# @title

df_monthly['f1_mllw_linear'] = df_monthly['trend_value'] + f1_mllw
df_monthly['f10_mllw_linear'] = df_monthly['trend_value'] + f10_mllw
df_monthly['f50_mllw_linear'] = df_monthly['trend_value'] + f50_mllw
df_monthly['f99_mllw_linear'] = df_monthly['trend_value'] + f99_mllw

# Isolate the last row of data for the specified current_year
projections_for_current_year = df_monthly.iloc[-1]
# Extract the MLLW-based AEP values for that year
f1_mllw_current = projections_for_current_year.get('f1_mllw_linear')
f10_mllw_current = projections_for_current_year.get('f10_mllw_linear')
f50_mllw_current = projections_for_current_year.get('f50_mllw_linear')
f99_mllw_current = projections_for_current_year.get('f99_mllw_linear')

epic_box_x = [epic_start, epic_end]  # X-axis range for the box
epic_box_y = [df_monthly['f1_mllw_linear'].min(),df_monthly['lowest'].max()]

fig = go.Figure()

fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['lowest'], mode='lines', name='monthly minima', line=dict(color='cornflowerblue')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f1_mllw_linear'], mode='lines', name='1% AEP', line=dict(color='red')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f10_mllw_linear'], mode='lines', name='10% AEP', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f50_mllw_linear'], mode='lines', name='50% AEP', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f99_mllw_linear'], mode='lines', name='99% AEP', line=dict(color='purple')))

fig.add_trace(go.Scatter(x=[epic_mid], y=[f1_mllw], mode='markers', name=epoch_info+' 1% AEP level', marker=dict(symbol='diamond', size=8, color='red', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f10_mllw], mode='markers', name=epoch_info+' 10% AEP level', marker=dict(symbol='square', size=8, color='orange', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f50_mllw], mode='markers', name=epoch_info+' 50% AEP level', marker=dict(symbol='triangle-up', size=8, color='blue', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f99_mllw], mode='markers', name=epoch_info+' 99% AEP level', marker=dict(symbol='circle', size=8, color='purple', line=dict(color='black', width=1)),showlegend=False))

fig.add_trace(go.Scatter(x=[current_year], y=[f1_mllw_current], mode='markers', name='2024 1% AEP level', marker=dict(symbol='diamond', size=12, color='red', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[current_year], y=[f10_mllw_current], mode='markers', name='2024 10% AEP level', marker=dict(symbol='square', size=12, color='orange', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[current_year], y=[f50_mllw_current], mode='markers', name='2024 50% AEP level', marker=dict(symbol='triangle-up', size=12, color='blue', line=dict(color='black', width=1)),showlegend=False))
fig.add_trace(go.Scatter(x=[current_year], y=[f99_mllw_current], mode='markers', name='2024 99% AEP level', marker=dict(symbol='circle', size=12, color='purple', line=dict(color='black', width=1)),showlegend=False))

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')

fig.update_layout(
    title={
        'text': 'Monthly Extreme Water Levels<br>'+station_info+'<br>'+trend_text,
        'x': 0.5,
        'y': 0.95,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis=dict(title='Year'),
    yaxis=dict(title='Meters below MLLW'),
    legend=dict(x=1, y=0.5, orientation='v'),
    margin=dict(l=0, r=0, t=50, b=0),
    showlegend=True,
    hovermode='closest',
    hoverlabel=dict(bgcolor="white", font_size=12),
    plot_bgcolor='rgba(255, 255, 255, 1)',  # White gridded background
    paper_bgcolor='rgba(255, 255, 255, 1)',
    width=900,  # Set width of the plot
    height=300,
    title_x=0.5,
        shapes=[
        dict(
            type="rect",
            xref="x",
            yref="y",
            x0=epic_box_x[0],
            y0=epic_box_y[0],
            x1=epic_box_x[1],
            y1=epic_box_y[1],
            fillcolor="gray",
            opacity=0.3,
            layer="below",
            line=dict(color="gray", width=1),
        )]
)

# Add dummy trace for legend entry of the shape
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=0),
    showlegend=True,
    legendgroup="shape_legend",
    name="Station Epoch",  # Name displayed in legend
    line=dict(color="gray", width=1),
    legendrank=1,
    fill='toself',  # Filled area of the shape
    fillcolor='gray',  # Fill color of the shape
))


fig.show()


## Seasonal Variation of Exceedance Probability Levels

Seasonal variation of the 1%, 10%, 50%, and 99% exceedance probability levels, shown in red, orange, blue and purple, respectively. Values, displayed in meters and feet, are relative to the Mean Higher High Water (MHHW), Mean Lower Low Water (MLLW), or Mean Sea Level (MSL) datums. The monthly exceedance probability levels were determined by 12 separate extreme value analyses carried out with only the extreme values of the specified month in each year. These levels indicate the months of the year when more extreme levels are likely.

In [202]:
# @title Seasonal variation of Extreme Water Levels

# --- Define the desired return periods and their colors ---
color_map = {
    100.0: 'red',
    10.0: 'gold', # Using 'gold' for better visibility than 'yellow'
    2.0: 'blue',
    1.01: 'purple'
}
# We will iterate through the keys of this dictionary to plot only these periods
periods_to_plot = list(color_map.keys())


# --- Create the Subplots ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("High Water Levels by Month", "Low Water Levels by Month")
)

if units == 'english':
    unit_label = 'feet'
else:
    unit_label = 'meters'

# --- Plot High Water Levels (Subplot 1) ---
for rp in periods_to_plot:
    df_period = seasonal_hi[seasonal_hi['returnPeriod'] == rp].sort_values('month')
    if not df_period.empty: # Only plot if the data exists
        prob = df_period['probability'].iloc[0]
        color = color_map[rp]

        fig.add_trace(go.Scatter(
            x=df_period['month'],
            y=df_period['level'],
            mode='lines+markers',
            name=f'{rp}-year RP ({prob*100:.0f}%)',
            line=dict(color=color) # Apply the specified color
        ), row=1, col=1)

# --- Plot Low Water Levels (Subplot 2) ---
for rp in periods_to_plot:
    df_period = seasonal_lo[seasonal_lo['returnPeriod'] == rp].sort_values('month')
    if not df_period.empty:
        prob = df_period['probability'].iloc[0]
        color = color_map[rp]

        fig.add_trace(go.Scatter(
            x=df_period['month'],
            y=df_period['level'],
            mode='lines+markers',
            name=f'{rp}-year RP ({prob*100:.0f}%)',
            line=dict(color=color),
            showlegend=False
        ), row=2, col=1)

# --- Update Layout and Axes ---
fig.update_layout(
    title_text='Monthly Exceedance Levels by Return Period',
    height=700,
    legend_title="Return Period (Probability)",
        plot_bgcolor='white',  # Set the plotting area background to white
    paper_bgcolor='white'  # Set the entire figure background to white
)
fig.update_yaxes(title_text=f"High Level ({unit_label} above MHHW)", row=1, col=1)
fig.update_yaxes(title_text=f"Low Level ({unit_label} above MHHW)", row=2, col=1)
fig.update_xaxes(
    title_text="Month",
    tickvals=np.arange(1, 13),
    ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    row=2, col=1
)

fig.show()

## **Step 6**
## Exploring [2022 Sea Level Rise Tech Report](https://oceanservice.noaa.gov/hazards/sealevelrise/sealevelrise-tech-report-sections.html) Extreme Water Level Analysis

This EWL information is based on an regional frequency analysis (RFA) of NOAA tide gauges within a 400-km radius of the center of each individual 1-degree grid and fit with a Generalized Pareto Distribution (GPD) of threshold exceedances. RFA is a statistical method used to estimate the frequency distribution of extreme events at locations where limited or no data is available. It involves pooling data from multiple sites within a region to develop regional frequency curves or equations that can be used to estimate the frequency of extreme events at ungauged sites.



In [203]:
# @title

# Convert the 'rl' column from text (e.g., '10yr') to a numeric value
myrfa['ARI'] = myrfa['rl'].str.replace('yr', '').astype(float)

high_y = myrfa['ci_high'].max()
if pd.isna(high_y): # Fallback if there are no high CI values
    high_y = myrfa['value'].max()

# Create a separate DataFrame for the confidence interval, removing missing values
myrfa_ci = myrfa.dropna(subset=['ci_low', 'ci_high']).copy()

# --- Plotting ---

# Create figure
fig = go.Figure()

# Add vertical lines for key ARIs
ari_lines = [
    {'x': 1, 'color': 'purple', 'label': '1-Year ARI'},
    {'x': 2, 'color': 'blue', 'label': '2-Year ARI'},
    {'x': 10, 'color': 'orange', 'label': '10-Year ARI'}, # Note: Changed yellow to orange for better visibility
    {'x': 50, 'color': 'pink', 'label': '50-Year ARI'},
    {'x': 100, 'color': 'red', 'label': '100-Year ARI'}
]

for line in ari_lines:
    fig.add_trace(go.Scatter(
        x=[line['x'], line['x']],
        y=[0, high_y],
        mode='lines',
        line=dict(color=line['color'], width=2),
        name=line['label']
    ))

# Fill between ci_low and ci_high using the cleaned data
fig.add_trace(go.Scatter(
    x=myrfa_ci['ARI'].tolist() + myrfa_ci['ARI'].tolist()[::-1],
    y=myrfa_ci['ci_low'].tolist() + myrfa_ci['ci_high'].tolist()[::-1],
    fill='toself',
    fillcolor='lightblue',
    opacity=0.5,
    line=dict(color='rgba(255,255,255,0)'), # Make the line itself invisible
    showlegend=False,
    name='Confidence Interval'
))

# Plot the main RFA Exceedance Level line and markers
fig.add_trace(go.Scatter(x=myrfa['ARI'], y=myrfa['value'], mode='lines', line=dict(color='blue'), name='RFA Exceedance Level'))

# --- Figure Layout and Styling ---

# Set y-axis range dynamically
high_y = myrfa['ci_high'].max()
if pd.isna(high_y): # Fallback if there are no high CI values
    high_y = myrfa['value'].max()
high_y *= 1.1 # Add 10% padding to the top

yaxis_title = 'feet above MHHW' if units == 'english' else 'meters above MHHW'

fig.update_layout(
    title=f'Regional Frequency Analysis<br>{station_info}',
    xaxis=dict(
        title='Average Recurrence Interval (Years)',
        type='log',
        tickvals=[0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100],
        ticktext=['0.1','0.2','0.5','1','2','5','10','20','50','100']
    ),
    yaxis=dict(
        title=yaxis_title,
        range=[0, high_y], # Set the y-axis range
        showgrid=True, gridwidth=1, gridcolor='lightgray',
        zeroline=True, zerolinewidth=2, zerolinecolor='black'
    ),
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.7)'),
    margin=dict(l=50, r=20, t=60, b=50),
    showlegend=True,
    hovermode='x unified',
    hoverlabel=dict(bgcolor="white", font_size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=300,
    title_x=0.5
)

fig.show()

In [204]:

# @title RFA in terms of Annual Exceedance Probability (AEP)

#  Create the numeric 'ARI' column
myrfa['ARI'] = myrfa['rl'].str.replace('yr', '').astype(float)

# Apply Gumbel transformation to create the 'AEP' column
# This formula converts Average Recurrence Interval to Annual Exceedance Probability
myrfa['AEP'] = (1 - np.exp(-1 / myrfa['ARI'])) * 100

# Create a clean DataFrame for the confidence interval
myrfa_ci = myrfa.dropna(subset=['ci_low', 'ci_high']).copy()

# Plotting with AEP ---
fig = go.Figure()

# Fill between ci_low and ci_high, using AEP for the x-axis
fig.add_trace(go.Scatter(
    x=myrfa_ci['AEP'].tolist() + myrfa_ci['AEP'].tolist()[::-1],
    y=myrfa_ci['ci_low'].tolist() + myrfa_ci['ci_high'].tolist()[::-1],
    fill='toself',
    fillcolor='lightblue',
    opacity=0.5,
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False,
    name='95% Confidence Interval'
))

# Plot the main RFA Exceedance Level, using AEP for the x-axis
fig.add_trace(go.Scatter(x=myrfa['AEP'], y=myrfa['value'], mode='lines', line=dict(color='blue'), name='RFA Exceedance Level'))

# Add vertical lines for key probabilities
high_y = myrfa['ci_high'].max()
if pd.isna(high_y):
    high_y = myrfa['value'].max()
high_y *= 1.1

prob_lines = [
    {'prob': 1.0, 'color': 'red', 'label': '1% AEP (100-year)'},
    {'prob': 2.0, 'color': 'lightcoral', 'label': '2% AEP (50-year)'},
    {'prob': 10, 'color': 'orange', 'label': '10% AEP (10-year)'},
    {'prob': 50, 'color': 'blue', 'label': '50% AEP (2-year)'},
    {'prob': 99, 'color': 'purple', 'label': '99% AEP (1-year)'}
]

for line in prob_lines:
    fig.add_trace(go.Scatter(
        x=[line['prob'], line['prob']],
        y=[0, high_y],
        mode='lines',
        line=dict(color=line['color'], width=2),
        name=line['label']
    ))

# --- Figure Layout and Styling ---
yaxis_title = 'feet above MHHW' if units == 'english' else 'meters above MHHW'

fig.update_layout(
    title=f'Regional Frequency Analysis Exceedance Probability Curve<br>{station_info}',
    xaxis=dict(
        title='Annual Probability of Exceedance (%)',
        type='log',
        autorange='reversed', # Reverses the axis to show low probability (rare events) on the right
        tickvals=[100, 90, 80, 50, 20, 10, 5, 2, 1, 0.5, 0.2],
        ticktext=['100','90','80','50','20','10','5','2','1','0.5','0.2']
    ),
    yaxis=dict(
        title=yaxis_title,
        range=[0, high_y],
        showgrid=True, gridwidth=1, gridcolor='lightgray',
        zeroline=True, zerolinewidth=2, zerolinecolor='black'
    ),
    legend=dict(x=1.01, y=0.95),
    margin=dict(l=50, r=20, t=60, b=50),
    showlegend=True,
    hovermode='x unified',
    hoverlabel=dict(bgcolor="white", font_size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=900,
    height=300,
    title_x=0.5
)

fig.show()

## Extreme Water Level plot using RFA-based methodology

The RFA results were computed from daily max water levels which were first detrended, then passed through a filtering function to find the max water level in a +/- 2 day window. These filtered values were then used to calculate the local u threshold which is the 98% percentile of the filtered data. All of the daily max values over the 98th percentile were then fit to the Generalized Pareto Distribution (GPD).

Below we plot the RFA derived ARI values with the daily max wl, an example of filtered daily maxes using our own filtering function, and the u-value with the historical linear trend.

*** Daily max is calculated from hourly observations while top ten water levels are calculated from 6 min data (when available). Due to these differences, top ten water levels may not line up exactly with the daily max values plotted below.

In [221]:
# @title
try:
  rfa_levels = myrfa

  daily_max_df = daily_max_df[daily_max_df['pcCompleteHourly']>50]
  daily_max_df['maxWL'] = pd.to_numeric(daily_max_df['valueHourly'],errors='coerce')
  daily_max_df['day'] = pd.to_datetime(daily_max_df['dateHourly'])

  # Helper function to find the closest value
  def get_value_for_closest_aep(myrfa_df, target_aep):
      """
      Finds the row where AEP is closest to the target_aep
      and returns the corresponding 'value'.
      """
      # Find the index of the row with the minimum absolute difference
      closest_index = (myrfa_df['AEP'] - target_aep).abs().idxmin()
      # Get the value from that row
      return myrfa_df.loc[closest_index, 'value']

  # Use the function to get the values for each percentage
  percent_99 = get_value_for_closest_aep(myrfa, 99)
  percent_50 = get_value_for_closest_aep(myrfa, 50)
  percent_10 = get_value_for_closest_aep(myrfa, 10)
  percent_2 = get_value_for_closest_aep(myrfa, 2)

  # Print the results to verify
  print(f"Value for AEP closest to 99%: {percent_99:.2f}")
  print(f"Value for AEP closest to 50%: {percent_50:.2f}")
  print(f"Value for AEP closest to 10%: {percent_10:.2f}")
  print(f"Value for AEP closest to 2%: {percent_2:.2f}")

  #python function for filtering data to find max in a period of days: a true max of an event
  def filter_data(df, r):
    #Step 1: Sort the data by the highest water level
    df_sort = df.sort_values(by='maxWL',ascending=False,na_position='last')
    #Step 2: Iterate through each row of the sorted data
    for j in range(len(df_sort)):
        #Step 3: Check if the second column value is not NaN
      if not np.isnan(df_sort['maxWL'].iloc[j]):
          #Step 4: Define the time window for the current row
        sub = (df_sort['day'] >= (df_sort['day'].iloc[j]-pd.to_timedelta(r,unit='D')))& (df_sort['day']<=  (df_sort['day'].iloc[j]+pd.to_timedelta(r, unit='D')))
    #Step 5: Check if there is more than one data point within the time window
        if sub.sum() > 1:
          #Step 5: Get the indices of data points within the time window
          id = sub[sub].index.tolist()
          #Step 6: Find the index of the maximum value within the time window
          id1 = df_sort.loc[id,'maxWL'].idxmax()
            #Step 7: Remove the index of the max water level from the list
          id.remove(id1)
            #Step 8: Assign NaN to all water levels that are not the max within the time frame
          df_sort.loc[id, 'maxWL'] = np.nan
    #Step 9: Remove rows with NaN values in the wl column (filtered data)
    fdata = df_sort.dropna(subset=['maxWL'])
  #Step 10: Retun the filtered data
    return fdata

  dailyMAX = daily_max_df[['day','maxWL']]

  r=2

  daily_filtered = filter_data(dailyMAX,r)

  local_u_midpoint = local_u_station.iloc[0]

  daily_max_df = daily_max_df.sort_values(by='day')

  daily_max_df['decimal_date']=daily_max_df['day'].dt.year+(daily_max_df['day'].dt.dayofyear -1)/365.2425

  # Handles cases with a single, continuous trend
  # Calculate a single base y-intercept for a trend line that passes through y=0 at the epoch's midpoint.
  y_int_base = getYInt(epic_mid, 0, mytrendrate)

  # Calculate the base linear trend across all dates.
  daily_max_df['linear_trend'] = mytrendrate * daily_max_df['decimal_date'] + y_int_base

  daily_max_df['trend_value'] = daily_max_df['linear_trend']
  print(daily_max_df)

  daily_max_df.loc[daily_max_df['day'] < trendstartdate, 'trend_value'] = np.nan
  daily_max_df.loc[daily_max_df['day'] > trendenddate, 'trend_value'] = np.nan

  # Calculate the percent
  daily_max_df['f2_mhhw_linear'] = daily_max_df['trend_value'] + percent_2
  daily_max_df['f10_mhhw_linear'] = daily_max_df['trend_value'] + percent_10
  daily_max_df['f50_mhhw_linear'] = daily_max_df['trend_value'] + percent_50
  daily_max_df['f99_mhhw_linear'] = daily_max_df['trend_value'] + percent_99
  daily_max_df['u_trend'] = daily_max_df['trend_value'] + local_u_station.iloc[0]

  # Grab the last calculated value from each respective trendline column
  f1_current = df_monthly['f1_mhhw_linear'].iloc[-1]
  f10_current = df_monthly['f10_mhhw_linear'].iloc[-1]
  f50_current = df_monthly['f50_mhhw_linear'].iloc[-1]
  f99_current = df_monthly['f99_mhhw_linear'].iloc[-1]
  excess = daily_max_df[daily_max_df['valueHourly'] > daily_max_df['u_trend']]
except:
  print('No RFA data for this station')

Value for AEP closest to 99%: 0.44
Value for AEP closest to 50%: 0.79
Value for AEP closest to 10%: 1.17
Value for AEP closest to 2%: 1.53
check
check2
check3
check4
check5
check6
       dateHourly timeHourly  valueHourly  pcCompleteHourly flagHourly  maxWL  \
36     1927-08-27      02:00        0.171               100          0  0.171   
83     1927-10-13      16:00        0.019               100          0  0.019   
235    1928-04-24      05:00        0.049               100          0  0.049   
249    1928-05-08      16:00        0.140               100          0  0.140   
250    1928-05-09      05:00        0.201               100          0  0.201   
...           ...        ...          ...               ...        ...    ...   
35313  2025-12-26      06:00        0.223               100          0  0.223   
35314  2025-12-27      19:00        0.365               100          0  0.365   
35315  2025-12-28      08:00        0.237               100          0  0.237   
35316  202

In [222]:
# @title Click off the blue "Daily Max above MHHW" line to see only the events at this station that were used in the RFA analysis
try:
  daily_max_df=daily_max_df[daily_max_df['maxWL']>0]
  daily_max_df=daily_max_df.sort_values(by='day',ascending=False)

  daily_max_trace = go.Scatter(x=daily_max_df['day'], y=daily_max_df['maxWL'], mode='lines', name='Daily Max above MHHW')
  filtered_trace = go.Scatter(x=daily_filtered['day'], y=daily_filtered['maxWL'], mode='markers', name='Filtered Data', line=dict(color='orange'))
  excess_trace = go.Scatter(x=excess['day'], y=excess['maxWL'], mode='markers', name='Filtered Data above threshold',line=dict(color='green'))

  return_periods_traces = []
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f2_mhhw_linear'], mode='lines', name='50 year ARI', line=dict(color='lightcoral')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f10_mhhw_linear'], mode='lines', name='10 year ARI', line=dict(color='orange')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f50_mhhw_linear'], mode='lines', name='2 year ARI', line=dict(color='blue')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f99_mhhw_linear'], mode='lines', name='1 year ARI', line=dict(color='purple')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['u_trend'], mode='lines', name=f'Threshold (u={local_u_midpoint})', line=dict(color='black', width=3)))


  top_ten_trace = go.Scatter(x=mytop10['date'], y=mytop10['height'], mode='markers', name='Top 10 Water Levels', marker=dict(color='black', size=10, symbol='x'))

  fig = go.Figure(data=[daily_max_trace, excess_trace, top_ten_trace] + return_periods_traces)
  fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
  fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')
  #fig.update_yaxes(range=[-0.5, 1.5])
  fig.update_layout(
      title=str(station) + '   RFA analysis',
      xaxis_title='Date',
      yaxis_title='meters above MHHW',
      legend=dict(x=1, y= 0.5, orientation='v', bgcolor='rgba(0,0,0,0)'),
      margin=dict(l=0, r=0, t=50, b=0),
      hovermode='x',
      plot_bgcolor='rgba(255, 255, 255, 1)',  # White gridded background
      paper_bgcolor='rgba(255, 255, 255, 1)',
      width=900,  # Set width of the plot
      height=400,
      title_x=0.5# White gridded background
  )

  if units == 'english':
    fig.update_layout(yaxis=dict(title='feet above MHHW'))
  # Show plot
  fig.show()
except:
  print('No RFA data for this station')

In [223]:
# @title Adding the names of the Top Ten events
try:
  daily_max_trace = go.Scatter(x=daily_max_df['day'], y=daily_max_df['maxWL'], mode='lines', name='Daily Max above MHHW')
  filtered_trace = go.Scatter(x=daily_filtered['day'], y=daily_filtered['maxWL'], mode='markers', name='Filtered Data', line=dict(color='orange'))
  excess_trace = go.Scatter(x=excess['day'], y=excess['maxWL'], mode='markers', name='Filtered Data above threshold',line=dict(color='green'))

  return_periods_traces = []
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f2_mhhw_linear'], mode='lines', name='50 year ARI', line=dict(color='lightcoral')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f10_mhhw_linear'], mode='lines', name='10 year ARI', line=dict(color='orange')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f50_mhhw_linear'], mode='lines', name='2 year ARI', line=dict(color='blue')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['f99_mhhw_linear'], mode='lines', name='1 year ARI', line=dict(color='purple')))
  return_periods_traces.append(go.Scatter(x=daily_max_df['day'], y=daily_max_df['u_trend'], mode='lines', name=f'Threshold (u={local_u_midpoint})', line=dict(color='black', width=3)))

  top_ten_trace = go.Scatter(x=mytop10['date'], y=mytop10['height'], mode='markers', name='Top 10 Water Levels', marker=dict(color='black', size=10, symbol='x'))

  fig = go.Figure(data=[daily_max_trace, excess_trace, top_ten_trace] + return_periods_traces)
  fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
  fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')

  # Add text annotations
  for i in range(len(mytop10)):
      fig.add_annotation(
          x=mytop10['date'].iloc[i],
          y=mytop10['height'].iloc[i] + 0.1,
          text=mytop10['event'].iloc[i],
          showarrow=False
      )

  fig.update_layout(
      title=str(station) + '\nRFA analysis',
      xaxis_title='Date',
      yaxis_title='Water Level (m) above MHHW',
      legend=dict(x=1, y= 0.5, orientation='v', bgcolor='rgba(0,0,0,0)'),
      margin=dict(l=0, r=0, t=50, b=0),
      hovermode='x',
      plot_bgcolor='rgba(255, 255, 255, 1)',  # White gridded background
      paper_bgcolor='rgba(255, 255, 255, 1)',
      width=900,  # Set width of the plot
      height=400,
      title_x=0.5# White gridded background
  )

  if units == 'english':
    fig.update_layout(yaxis=dict(title='feet above MHHW'))
  # Show plot
  fig.show()
except:
  print('No RFA data for this station')


## **Step 7**
## Comparison of methods

### It's important to note that both the GEV and RFA methodologies have specific use cases where one may be more advantageous than the other. CO-OPS GEV relies solely on data from the station of interest. This means the method can be particularly useful when there is a long time series of data available, but it may be less effective with less than 30 years of data. On the other hand, RFA is based on a broader regional perspective, which can be beneficial when a station lacks significant data but fits into a regional pattern. However, it may be less effective when a station is distinct from its surrounding neighbors.

In [232]:
# @title
try:
  # Prepare Data and Figure
  fig = go.Figure()

  # Add Traces for GEV (Station-Specific Analysis)

  # GEV 90% Confidence Interval (light gray fill)
  fig.add_trace(go.Scatter(
      x=mydf_hi['probability'].tolist() + mydf_hi['probability'].tolist()[::-1],
      y=mydf_hi['upper90CI'].tolist() + mydf_hi['lower90CI'].tolist()[::-1],
      fill='toself',
      opacity=0.5,
      fillcolor='lightgray',
      line=dict(color='rgba(255,255,255,0)'),
      name='GEV 90% Confidence Interval',
      hoverinfo='skip' # Hide hover info for the fill
  ))

  # GEV Exceedance Level line
  fig.add_trace(go.Scatter(
      x=mydf_hi['probability'],
      y=mydf_hi['level'],
      mode='lines',
      line=dict(color='black', width=2),
      name='GEV Exceedance Level'
  ))


  # Add Traces for RFA (Regional Analysis)

  # RFA 95% Confidence Interval (light blue fill)
  fig.add_trace(go.Scatter(
      x=myrfa_ci['AEP'].tolist() + myrfa_ci['AEP'].tolist()[::-1],
      y=myrfa_ci['ci_low'].tolist() + myrfa_ci['ci_high'].tolist()[::-1],
      fill='toself',
      fillcolor='lightblue',
      opacity=0.5,
      line=dict(color='rgba(255,255,255,0)'),
      name='RFA 95% Confidence Interval',
      hoverinfo='skip'
  ))

  # RFA Exceedance Level line
  fig.add_trace(go.Scatter(
      x=myrfa['AEP'],
      y=myrfa['value'],
      mode='lines',
      line=dict(color='blue', width=2),
      name='RFA Exceedance Level'
  ))


  # Add Vertical Probability Lines

  # Calculate the maximum y-value from all data to set the line height
  all_y_values = pd.concat([
      myrfa_ci['ci_high'],
      mydf_hi['upper90CI']
  ])
  high_y = all_y_values.max() * 1.1 # Add 10% buffer

  # Define and draw the lines
  prob_lines = [
      {'prob': 1.0, 'color': 'red', 'label': '1% AEP (100-year)'},
      {'prob': 2.0, 'color': 'lightcoral', 'label': '2% AEP (50-year)'},
      {'prob': 10, 'color': 'orange', 'label': '10% AEP (10-year)'},
      {'prob': 50, 'color': 'blue', 'label': '50% AEP (2-year)'},
      {'prob': 99.0, 'color': 'purple', 'label': '99% AEP (~1-year)'}
  ]

  for line in prob_lines:
      fig.add_trace(go.Scatter(
          x=[line['prob'], line['prob']],
          y=[0, high_y],
          mode='lines',
          line=dict(color=line['color'], width=1.5),
          name=line['label']
      ))

  # Final Figure Layout and Styling
  yaxis_title = 'feet above MHHW' if units == 'english' else 'meters above MHHW'

  fig.update_layout(
      title=f'GEV and RFA Exceedance Probability Curves<br>{station_info}',
      xaxis=dict(
          title='Annual Probability of Exceedance (%)',
          type='log',
          autorange='reversed',
          tickvals=[100, 90, 80, 70, 60, 50, 40, 30, 20, 10, 5, 2, 1, 0.5, 0.2],
          ticktext=['100','90','80','70','60','50','40','30','20','10','5','2','1','0.5','0.2']
      ),
      yaxis=dict(
          title=yaxis_title,
          range=[0, high_y],
          showgrid=True, gridwidth=1, gridcolor='lightgray',
          zeroline=True, zerolinewidth=2, zerolinecolor='black'
      ),
      legend=dict(x=1.01, y=0.95),
      margin=dict(l=50, r=20, t=60, b=50),
      showlegend=True,
      hovermode='x unified',
      hoverlabel=dict(bgcolor="white", font_size=12),
      plot_bgcolor='white',
      paper_bgcolor='white',
      width=900,
      height=500,
      title_x=0.5
  )

  fig.show()
except:
  print('No RFA data for this station')

## **Step 8**
## Future Extreme Water Levels

### High and low annual exceedance probability levels are shown relative to the tidal datums and the geodetic North American Vertical Datum (NAVD88), if available. The levels are in meters relative to the National Tidal Datum Epoch (1983-2001) Mean Sea Level datum at most stations or a recent 5-year modified epoch MSL datum at stations with rapid sea level rates in Louisiana, Texas, and Alaska. On the left are the CO-OPS exceedance probability levels for the mid-year of the tidal epoch currently in effect for the station. In the middle are the CO-OPS projected exceedance probability levels and tidal datums assuming continuation of the linear historic trend to 2025. On the right are the projected exceedance probability levels and tidal datums assuming continuation of the linear historic trend to date of choice.

In [209]:
# @title Choose a future date

#choose a future year to plot
future_year = 2050

In [210]:
# @title

# Calculate projections for the current year and round to 2 decimal points
f1_current_msl = round(f1_mhhw_msl + (mytrendrate * (current_year - epic_mid)), 2)
f10_current_msl = round(f10_mhhw_msl + (mytrendrate * (current_year - epic_mid)), 2)
f50_current_msl = round(f50_mhhw_msl + (mytrendrate * (current_year - epic_mid)), 2)
f99_current_msl = round(f99_mhhw_msl + (mytrendrate * (current_year - epic_mid)), 2)

f1_mllw_current_msl = round(f1_mllw_msl + (mytrendrate * (current_year - epic_mid)), 2)
f10_mllw_current_msl = round(f10_mllw_msl + (mytrendrate * (current_year - epic_mid)), 2)
f50_mllw_current_msl = round(f50_mllw_msl + (mytrendrate * (current_year - epic_mid)), 2)
f99_mllw_current_msl = round(f99_mllw_msl + (mytrendrate * (current_year - epic_mid)), 2)

mhhw_current = round(MHHW_msl_datum + (mytrendrate * (current_year - epic_mid)), 2)
mllw_current = round(MLLW_msl_datum + (mytrendrate * (current_year - epic_mid)), 2)
mhw_current = round(MHW_msl_datum + (mytrendrate * (current_year - epic_mid)), 2)
mlw_current = round(MLW_msl_datum + (mytrendrate * (current_year - epic_mid)), 2)


# --- Calculate Future Year Projections (Rounded) ---
f1_future_year_msl = round(f1_mhhw_msl + (mytrendrate * (future_year - epic_mid)), 2)
f10_future_year_msl = round(f10_mhhw_msl + (mytrendrate * (future_year - epic_mid)), 2)
f50_future_year_msl = round(f50_mhhw_msl + (mytrendrate * (future_year - epic_mid)), 2)
f99_future_year_msl = round(f99_mhhw_msl + (mytrendrate * (future_year - epic_mid)), 2)

f1_mllw_future_year_msl = round(f1_mllw_msl + (mytrendrate * (future_year - epic_mid)), 2)
f10_mllw_future_year_msl = round(f10_mllw_msl + (mytrendrate * (future_year - epic_mid)), 2)
f50_mllw_future_year_msl = round(f50_mllw_msl + (mytrendrate * (future_year - epic_mid)), 2)
f99_mllw_future_year_msl = round(f99_mllw_msl + (mytrendrate * (future_year - epic_mid)), 2)

mhhw_future_year = round(MHHW_msl_datum + (mytrendrate * (future_year - epic_mid)), 2)
mllw_future_year = round(MLLW_msl_datum + (mytrendrate * (future_year - epic_mid)), 2)
mhw_future_year = round(MHW_msl_datum + (mytrendrate * (future_year - epic_mid)), 2)
mlw_future_year = round(MLW_msl_datum + (mytrendrate * (future_year - epic_mid)), 2)


# --- Create Figure and Add Consolidated Traces ---
fig = go.Figure()

# Define the labels and positions for the x-axis
x_values = [1, 2, 3]
x_labels = [epoch_info, str(current_year), str(future_year)]

# Add AEP (Exceedance) Traces
# 99%
fig.add_trace(go.Scatter(x=x_values, y=[f99_mhhw_msl, f99_current_msl, f99_future_year_msl], mode='markers', name='99% AEP High', legendgroup='99', marker=dict(symbol='circle', color='purple', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[f99_mllw_msl, f99_mllw_current_msl, f99_mllw_future_year_msl], mode='markers', name='99% AEP Low', legendgroup='99', marker=dict(symbol='circle', color='purple', size=10)))
# 50%
fig.add_trace(go.Scatter(x=x_values, y=[f50_mhhw_msl, f50_current_msl, f50_future_year_msl], mode='markers', name='50% AEP High', legendgroup='50', marker=dict(symbol='triangle-up', color='blue', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[f50_mllw_msl, f50_mllw_current_msl, f50_mllw_future_year_msl], mode='markers', name='50% AEP Low', legendgroup='50', marker=dict(symbol='triangle-up', color='blue', size=10)))
# 10%
fig.add_trace(go.Scatter(x=x_values, y=[f10_mhhw_msl, f10_current_msl, f10_future_year_msl], mode='markers', name='10% AEP High', legendgroup='10', marker=dict(symbol='square', color='orange', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[f10_mllw_msl, f10_mllw_current_msl, f10_mllw_future_year_msl], mode='markers', name='10% AEP Low', legendgroup='10', marker=dict(symbol='square', color='orange', size=10)))
# 1%
fig.add_trace(go.Scatter(x=x_values, y=[f1_mhhw_msl, f1_current_msl, f1_future_year_msl], mode='markers', name='1% AEP High', legendgroup='1', marker=dict(symbol='diamond', color='red', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[f1_mllw_msl, f1_mllw_current_msl, f1_mllw_future_year_msl], mode='markers', name='1% AEP Low', legendgroup='1', marker=dict(symbol='diamond', color='red', size=10)))

# Add Tidal Datum Traces
fig.add_trace(go.Scatter(x=x_values, y=[MHHW_msl_datum, mhhw_current, mhhw_future_year], mode='markers', name='MHHW', marker=dict(symbol='x', color='black', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[MLLW_msl_datum, mllw_current, mllw_future_year], mode='markers', name='MLLW', marker=dict(symbol='x', color='black', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[MHW_msl_datum, mhw_current, mhw_future_year], mode='markers', name='MHW', marker=dict(symbol='cross', color='black', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[MLW_msl_datum, mlw_current, mlw_future_year], mode='markers', name='MLW', marker=dict(symbol='cross', color='black', size=10)))
fig.add_trace(go.Scatter(x=x_values, y=[NAVD88_msl_datum, NAVD88_msl_datum, NAVD88_msl_datum], mode='markers', name='NAVD88', marker=dict(symbol='bowtie', color='red', size=10)))


# --- Final Layout and Styling ---
fig.update_layout(
    title=f'Exceedance Probability Level and Tidal Datums<br>{station_info}',
    yaxis_title='Meters relative to MSL Datum',
    legend=dict(x=1, y=0.5, orientation='v'),
    width=800,
    height=800,
    title_x=0.5,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickmode='array',
        tickvals=x_values,
        ticktext=x_labels,
        range=[0.5, 3.5]
    )
)

# Apply gridlines and zerolines
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', zeroline=True, zerolinewidth=2, zerolinecolor='black')

# Handle unit conversion for y-axis title
if units == 'english':
  fig.update_layout(yaxis_title='Feet relative to MSL Datum')

fig.show()

In [211]:
# @title Extending out the historical linear trend

# Ensure there are no empty 'year' or 'month' values
df_monthly.dropna(subset=['year', 'month'], inplace=True)

# Create a reliable DatetimeIndex from the year and month columns
df_monthly.index = pd.to_datetime(df_monthly['year'].astype(int).astype(str) + '-' + df_monthly['month'].astype(int).astype(str))

# Find the last date in the historical data
last_month_dt = df_monthly.index.max()

# Create a date range for the future extension
start_date = last_month_dt + pd.DateOffset(months=1)
end_date = pd.to_datetime(f"{future_year}-12-31")
date_range = pd.date_range(start=start_date, end=end_date, freq='MS') # 'MS' for month-start frequency
df_extended = pd.DataFrame(index=date_range)

# Combine the historical and future DataFrames
df_final = pd.concat([df_monthly, df_extended])

# Calculate decimal_date from the final, complete index
df_final['decimal_date'] = df_final.index.year + (df_final.index.dayofyear - 1) / 365.2425

# Handles cases with a single, continuous trend
# Calculate a single base y-intercept for a trend line that passes through y=0 at the epoch's midpoint.
y_int_base = getYInt(epic_mid, 0, mytrendrate)

# Calculate the base linear trend across all dates.
df_final['linear_trend'] = mytrendrate * df_final['decimal_date'] + y_int_base

df_final['base_trend'] = df_final['linear_trend']
df_final.loc[df_final['date'] < trendstartdate, 'base_trend'] = np.nan
df_final.loc[df_final['date'] > trendenddate, 'base_trend'] = np.nan

# Datum-based trendlines
df_final['msl_linear'] = df_final['base_trend'] + (MSL_datum)
df_final['mhhw_linear'] = df_final['base_trend'] + (MHHW_msl_datum)
df_final['mllw_linear'] = df_final['base_trend'] + (MLLW_msl_datum)
df_final['mhw_linear'] = df_final['base_trend'] + (MHW_msl_datum)
df_final['mlw_linear'] = df_final['base_trend'] + (MLW_msl_datum)
df_final['navd88_linear'] = df_final['base_trend'] + (NAVD88_msl_datum)

# AEP trendlines relative to MHHW
df_final['f1_mhhw_linear'] = df_final['base_trend'] + (f1_mhhw)
df_final['f10_mhhw_linear'] = df_final['base_trend'] + (f10_mhhw)
df_final['f50_mhhw_linear'] = df_final['base_trend'] + (f50_mhhw)
df_final['f99_mhhw_linear'] = df_final['base_trend'] + (f99_mhhw)

# AEP trendlines relative to MLLW
df_final['f1_mllw_linear'] = df_final['base_trend'] + (f1_mllw)
df_final['f10_mllw_linear'] = df_final['base_trend'] + (f10_mllw)
df_final['f50_mllw_linear'] = df_final['base_trend'] + (f50_mllw)
df_final['f99_mllw_linear'] = df_final['base_trend'] + (f99_mllw)

# --- Get projections for the final year of the simulation (e.g., 2050) ---
latest_projections = df_final.iloc[-1]
f1_future_year = latest_projections.get('f1_mhhw_linear')
f10_future_year = latest_projections.get('f10_mhhw_linear')
f50_future_year = latest_projections.get('f50_mhhw_linear')
f99_future_year = latest_projections.get('f99_mhhw_linear')

# --- Get projections for the specific "current_year" (e.g., 2024) ---
projections_for_current_year = df_final[df_final.index.year == current_year].iloc[-1]
f1_2024 = projections_for_current_year.get('f1_mhhw_linear')
f10_2024 = projections_for_current_year.get('f10_mhhw_linear')
f50_2024 = projections_for_current_year.get('f50_mhhw_linear')
f99_2024 = projections_for_current_year.get('f99_mhhw_linear')

fig = go.Figure()

fig.add_trace(go.Scatter(x=df_final['decimal_date'], y=df_final['highest'], mode='lines', name='monthly maxima', line=dict(color='cornflowerblue')))
fig.add_trace(go.Scatter(x=df_final['decimal_date'], y=df_final['f1_mhhw_linear'], mode='lines', name='1% AEP', line=dict(color='red')))
fig.add_trace(go.Scatter(x=df_final['decimal_date'], y=df_final['f10_mhhw_linear'], mode='lines', name='10% AEP', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=df_final['decimal_date'], y=df_final['f50_mhhw_linear'], mode='lines', name='50% AEP', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=df_final['decimal_date'], y=df_final['f99_mhhw_linear'], mode='lines', name='99% AEP', line=dict(color='purple')))

fig.add_trace(go.Scatter(x=[epic_mid], y=[f1_mhhw], mode='markers', name=epoch_info+' 1% AEP level', marker=dict(symbol='diamond', size=8, color='red', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f10_mhhw], mode='markers', name=epoch_info+' 10% AEP level', marker=dict(symbol='square', size=8, color='orange', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f50_mhhw], mode='markers', name=epoch_info+' 50% AEP level', marker=dict(symbol='triangle-up', size=8, color='blue', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[epic_mid], y=[f99_mhhw], mode='markers', name=epoch_info+' 99% AEP level', marker=dict(symbol='circle', size=8, color='purple', line=dict(color='black', width=1))))

fig.add_trace(go.Scatter(x=[current_year], y=[f1_current], mode='markers', name='2024 1% AEP level', marker=dict(symbol='diamond', size=12, color='red', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[current_year], y=[f10_current], mode='markers', name='2024 10% AEP level', marker=dict(symbol='square', size=12, color='orange', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[current_year], y=[f50_current], mode='markers', name='2024 50% AEP level', marker=dict(symbol='triangle-up', size=12, color='blue', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[current_year], y=[f99_current], mode='markers', name='2024 99% AEP level', marker=dict(symbol='circle', size=12, color='purple', line=dict(color='black', width=1))))

fig.add_trace(go.Scatter(x=[future_year], y=[f1_future_year], mode='markers', name=str(future_year)+' 1% AEP level', marker=dict(symbol='diamond', size=12, color='red', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[future_year], y=[f10_future_year], mode='markers', name=str(future_year)+' 10% AEP level', marker=dict(symbol='square', size=12, color='orange', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[future_year], y=[f50_future_year], mode='markers', name=str(future_year)+' 50% AEP level', marker=dict(symbol='triangle-up', size=12, color='blue', line=dict(color='black', width=1))))
fig.add_trace(go.Scatter(x=[future_year], y=[f99_future_year], mode='markers', name=str(future_year)+' 99% AEP level', marker=dict(symbol='circle', size=12, color='purple', line=dict(color='black', width=1))))

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')
    #fig.update_yaxes(range=[-0.3, 1.5])
fig.update_layout(
        title={
            'text': 'Monthly Extreme Water Levels<br>'+station_info+'<br>'+trend_text,
            'x': 0.5,
            'y': 0.95,
            'xanchor': 'center',
            'yanchor': 'top'
        },
        xaxis=dict(title='Year'),
        yaxis=dict(title='Meters above MHHW'),
        legend=dict(x=1, y=0.5, orientation='v'),
        margin=dict(l=0, r=0, t=50, b=0),
        showlegend=True,
        hovermode='closest',
        hoverlabel=dict(bgcolor="white", font_size=12),
        plot_bgcolor='rgba(255, 255, 255, 1)',  # White gridded background
        paper_bgcolor='rgba(255, 255, 255, 1)',
        width=900,  # Set width of the plot
        height=300,
        title_x=0.5# White gridded background
    )

if units == 'english':
    fig.update_layout(yaxis=dict(title='feet above MHHW'))

fig.show()


In [212]:
# @title How will the extremes water levels change with future SL projections?

#choose a future scenario to plot
#options are Low, Intermediate-Low, Intermediate, Intermediate-High, High
future_scenario = 'Intermediate-High'

In [213]:
# @title Plot the future scenario

df_monthly = df_monthly[df_monthly['decimal_date']<2025]
replace_date = 2005
condition = df_monthly['decimal_date'] > replace_date
colums_to_replace = ['f1_mhhw_linear', 'f10_mhhw_linear', 'f50_mhhw_linear', 'f99_mhhw_linear']
df_monthly.loc[condition, colums_to_replace] = float('nan')
try:
  product = "slr_projections"
  server = "https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/"
  affil = "US"
  report_year='2022'
  myurl=(server+ product  + ".json?"+"units="+units+"&station=" + station + "&report_year=" + report_year+"&affil=" + affil)
  urlResponse = requests.get(myurl)
  content=urlResponse.json()
  # Assign data to variables
  mydata = content['SlrProjections']
  # Make it a Dataframe
  mydf_slr_projection = pd.DataFrame(mydata)
  mydf_slr_projection['day'] = '01'
  mydf_slr_projection['month'] = '01'
  mydf_slr_projection['year_date'] = pd.to_datetime(dict(year=mydf_slr_projection.projectionYear,
                                                            month=mydf_slr_projection.month,day=mydf_slr_projection.day))
  mydf_slr_projection=mydf_slr_projection[mydf_slr_projection['year_date']<'2051-01-01']

  # Break down SLR Projections by Scenario
  scenario=mydf_slr_projection[mydf_slr_projection['scenario']==future_scenario]
  scenario.reset_index(inplace=True,drop=True)
  if units == 'metric':
        #converting from cm to m
        scenario['projection']=scenario['projectionRsl']*0.01
  else:
        #converting from in to ft
        scenario['projection']=scenario['projectionRsl']*0.0833333

  product = "slr_projectionOffsets"
  server = "https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/"
  affil = "US"
  report_year='2022'
  myurl=(server+ product  + ".json?"+"units="+units+"&station=" + station + "&report_year=" + report_year+"&affil=" + affil)
  urlResponse = requests.get(myurl)
  content=urlResponse.json()
  # Assign data to variables
  mydata = content['SlrProjectionOffsets']

  product = "slr_projectionOffsets"
  server = "https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/"
  affil = "US"
  report_year='2022'
  myurl=(server+ product  + ".json?"+"units="+units+"&station=" + station + "&report_year=" + report_year+"&affil=" + affil)
  urlResponse = requests.get(myurl)
  content=urlResponse.json()
  # Assign data to variables
  mydata = content['SlrProjectionOffsets']
  mydf_slr_projection_offset = pd.DataFrame(mydata)
  offset = mydf_slr_projection_offset['offset'].sum()
  if units == 'metric':
    offset = offset * 0.01
  else:
    offset = offset * 0.0833333

  scenario['projection_MHHW'] = scenario['projection']+offset

  scenario['projection_MHHW_epic_mid'] = scenario['projection_MHHW']

  scenario['f99'] = f99_mhhw + scenario['projection_MHHW_epic_mid']
  scenario['f50'] = f50_mhhw + scenario['projection_MHHW_epic_mid']
  scenario['f10'] = f10_mhhw + scenario['projection_MHHW_epic_mid']
  scenario['f1'] = f1_mhhw + scenario['projection_MHHW_epic_mid']
  scenario['decimal_date']=scenario.year_date.dt.year+(scenario.year_date.dt.dayofyear -1)/365.2425

  fig = go.Figure()

  fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['highest'], mode='lines', name='monthly maxima', line=dict(color='cornflowerblue')))
  fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f1_mhhw_linear'], mode='lines', name='1% AEP', line=dict(color='red')))
  fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f10_mhhw_linear'], mode='lines', name='10% AEP', line=dict(color='orange')))
  fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f50_mhhw_linear'], mode='lines', name='50% AEP', line=dict(color='blue')))
  fig.add_trace(go.Scatter(x=df_monthly['decimal_date'], y=df_monthly['f99_mhhw_linear'], mode='lines', name='99% AEP', line=dict(color='purple')))

  fig.add_trace(go.Scatter(x=scenario['decimal_date'], y=scenario['f1'], mode='lines', name='1% AEP', line=dict(color='red')))
  fig.add_trace(go.Scatter(x=scenario['decimal_date'], y=scenario['f10'], mode='lines', name='10% AEP', line=dict(color='orange')))
  fig.add_trace(go.Scatter(x=scenario['decimal_date'], y=scenario['f50'], mode='lines', name='50% AEP', line=dict(color='blue')))
  fig.add_trace(go.Scatter(x=scenario['decimal_date'], y=scenario['f99'], mode='lines', name='99% AEP', line=dict(color='purple')))

  fig.add_trace(go.Scatter(x=[epic_mid], y=[f1_mhhw], mode='markers', name=epoch_info+' 1% AEP level', marker=dict(symbol='diamond', size=8, color='red', line=dict(color='black', width=1))))
  fig.add_trace(go.Scatter(x=[epic_mid], y=[f10_mhhw], mode='markers', name=epoch_info+' 10% AEP level', marker=dict(symbol='square', size=8, color='orange', line=dict(color='black', width=1))))
  fig.add_trace(go.Scatter(x=[epic_mid], y=[f50_mhhw], mode='markers', name=epoch_info+' 50% AEP level', marker=dict(symbol='triangle-up', size=8, color='blue', line=dict(color='black', width=1))))
  fig.add_trace(go.Scatter(x=[epic_mid], y=[f99_mhhw], mode='markers', name=epoch_info+' 99% AEP level', marker=dict(symbol='circle', size=8, color='purple', line=dict(color='black', width=1))))

  fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
  fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')
  #fig.update_yaxes(range=[-0.3, 1.5])
  fig.update_layout(
      title={
          'text': 'Monthly Extreme Water Levels<br>'+station_info+'<br>'+trend_text+'<br>'+future_scenario+" Future Scenario",
          'x': 0.5,
          'y': 0.95,
          'xanchor': 'center',
          'yanchor': 'top'
      },
      xaxis=dict(title='Year'),
      yaxis=dict(title='Meters above MHHW'),
      legend=dict(x=1, y=0.5, orientation='v'),
      margin=dict(l=0, r=0, t=50, b=0),
      showlegend=True,
      hovermode='closest',
      hoverlabel=dict(bgcolor="white", font_size=12),
      plot_bgcolor='rgba(255, 255, 255, 1)',  # White gridded background
      paper_bgcolor='rgba(255, 255, 255, 1)',
      width=900,  # Set width of the plot
      height=300,
      title_x=0.5# White gridded background
  )

  if units == 'english':
    fig.update_layout(yaxis=dict(title='feet above MHHW'))

  fig.show()
except:
  print("No future projections available for this station")

# Cumulative Risk plot

## Shows the cumulative risk (%) of inundation based on the 1% annual exceedance probability level with and without relative sea level change.

In [214]:
# @title Read in our ex_params.csv and format data

exparams = pd.read_csv('ex_params_2025.csv')

# Initialize plot variables. Plot runs from current year to 100 years later.
# Center, high, and low are the risk curve with error bounds
# for a linear relative sea level trend. Center0, high0, and low0, are the risk curve with error bounds for zero relative sea level change.
startyear = dt.datetime.now().year
numyr = 143
# Create dataframe for risk curves
cnames = ['station_id','scenario','year','center','high','low','center0','high0','low0']
risk = pd.DataFrame(index=range(numyr),columns=cnames)

year = list(range(startyear,startyear+numyr))
level = [0] * numyr
center = [0] * numyr
high = [0] * numyr
low = [0] * numyr
center0 = [0] * numyr
high0 = [0] * numyr
low0 = [0] * numyr

nams = ""
nums = 0
hiloc = 0
loloc = 0
hiscale = 0
loscale = 0
hishape = 0
loshape = 0

def to_feet(x):
    return x / 0.3048

def to_meters(x):
    return x * 0.3048

# Read in GEV location, scale, and shape parameters and error bounds for the selected station.
# Level is calculated as the current level for the 1% probability of exceedance of the station in the first year
# of the period assuming no relative sea level change since the station's epoch.
for j in range(len(exparams)):
    if str(exparams.station_id[j]) == station:
        nums = int(exparams.station_id[j])
        level = round(to_feet(exparams.exceed_location[j] + ((-math.log(0.99)) ** (-exparams.exceed_shape[j]) - 1) * (exparams.exceed_scale[j] / exparams.exceed_shape[j])), 1)
        level = to_meters(level)
        hiloc = exparams.exceed_location[j] + 1.28 * exparams.exceed_location_sd[j]
        loloc = exparams.exceed_location[j] - 1.28 * exparams.exceed_location_sd[j]
        hiscale = exparams.exceed_scale[j] + 1.28 * exparams.exceed_scale_sd[j]
        loscale = exparams.exceed_scale[j] - 1.28 * exparams.exceed_scale_sd[j]
        hishape = exparams.exceed_shape[j] + 1.28 * exparams.exceed_shape_sd[j]
        loshape = exparams.exceed_shape[j] - 1.28 * exparams.exceed_shape_sd[j]

# Add level column to exparams
exparams['level'] = level
exparams['hiloc'] = hiloc
exparams['loloc'] = loloc
exparams['hiscale'] = hiscale
exparams['loscale'] = loscale
exparams['hishape'] = hishape
exparams['loshape'] = loshape


In [215]:
# @title
# Initialize plot variables. Plot runs from current year to 100 years later.
start_year = dt.datetime.now().year
num_years = 142

# Create arrays for risk curves
num_years=101
year = np.arange(start_year, start_year + num_years)
center = np.zeros(num_years)
center0 = np.zeros(num_years)
high = np.zeros(num_years)
low = np.zeros(num_years)
high0 = np.zeros(num_years)
low0 = np.zeros(num_years)

for i in range(len(exparams)) :
    if str(exparams.station_id[i]) == station :
        paramstn = i

# Extract locations, scales, and shapes from exparams
# Get the default station's GEV parameters
locations = [exparams.exceed_location[paramstn],exparams.hiloc[paramstn],exparams.loloc[paramstn]]
scales = [exparams.exceed_scale[paramstn],exparams.hiscale[paramstn],exparams.loscale[paramstn]]
shapes = [exparams.exceed_shape[paramstn],exparams.hishape[paramstn],exparams.loshape[paramstn]]

yr = 0
numyr=len(year)
# Calculate the risk curves with and without relative sea level change for the 101 year period.
for yr in range(numyr) :
    risk.iloc[yr,0] = int(exparams.iloc[paramstn,0])
    risk.iloc[yr,1] = 'Historical Trend Projection'
    risk.iloc[yr,2] = int(year[yr])

for l in range(2):
    staTrend = mytrendrate
    if (l == 1):
      staTrend = 0
    for i in range(3) :
        for yr in range(0,numyr) :
            eq1 =  1 + shapes[i] * (level - staTrend  * (year[yr] - epic_mid) - locations[i]) / scales[i]
            if(eq1 < 0) : eq1 = 0
            eq2 =  np.exp(-((eq1) ** (-1 / shapes[i])))
            if(yr == 0) :
                risk.iloc[yr,i+3+l*3] = eq2 ** (yr+1)
            else:
                risk.iloc[yr,i+3+l*3] = (eq2) * risk.iloc[yr-1,i+3+l*3]

# Invert the probability of non-exceedance to give the probability of exceedance
for i in range(3,9) :
    for yr in range(numyr) :
        risk.iloc[yr,i] = 1 - risk.iloc[yr,i]
for yr1 in range(numyr) :
    risk.iloc[yr1,0] = int(exparams.station_id[paramstn])
    risk.iloc[yr1,1] = 'Historical Trend Projection'
    risk.iloc[yr1,2] = int(year[yr1])
risk=risk.dropna()


fig = go.Figure()

# Add risk curves and shaded areas to the plot
fig.add_trace(go.Scatter(x=np.concatenate([risk['year'], risk['year'][::-1]]),
                         y=np.concatenate([risk['high'], risk['low'][::-1]]),
                         fill='toself',opacity=0.5,fillcolor='orange', line=dict(color='rgba(255,255,255,0)'), name='80% Confidence Interval'))
fig.add_trace(go.Scatter(x=risk['year'], y=risk['center'], mode='lines', name='With relative sea level change', line=dict(color='red')))
fig.add_trace(go.Scatter(x=risk['year'], y=risk['center0'], mode='lines', name='No relative sea level change since tidal datum epoch', line=dict(color='green')))

# Round the y-axis labels to two decimal places
#rounded_y_labels = [round(label, 2) for label in fig.data[0].y]
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(zeroline=True, zerolinewidth=2, zerolinecolor='black')

# Customize layout
fig.update_layout(
    title=f'Change of at least one inundation at {round(level,2)} m above MHHW <br>{station_info}',
    title_x=0.5,
    # xaxis=dict(title='Last year for time period beginning in ' + str(spinner1.value)),
    yaxis=dict(title='Cumulative Risk', tickformat='%'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,  # Set width of the plot
    height=500
)
# Format hover template to round y values to 2 decimals
fig.update_traces(hovertemplate='Year: %{x}<br>Cumulative Risk: %{y:.2f}')
# Update y-axis to display values with 2 decimal places
fig.update_yaxes(tickformat='2%')

if units == 'english':
  level = level *3.28084
  fig.update_layout(title=f'Change of at least one inundation at {round(level,2)} ft above MHHW <br>{station_info}',title_x=0.5)
# Show the plot
fig.show()

## **Takeaways**

###Thank you for taking the time to walk through this Extremes Station Exploration notebook.

###Some key takeaways include:

*   How to access CO-OPS data from the CO-OPS API (here is a [URL builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html) if you want to explore other CO-OPS data)
*   Understanding different types of extreme analysis and the value and use cases for each method
*   Exploring how extreme values relate to tidal datums
*   Understanding and plotting the relationship between extreme water levels and sea level change
